In [1]:
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg
import pandas as pd

[nltk_data] Downloading package gutenberg to /home/s4ic0/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.


In [4]:

data=gutenberg.raw('shakespeare-hamlet.txt')

with open('hamlet.txt','w') as file:
    file.write(data)

In [5]:
#data preprocessing

import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [13]:
with open('hamlet.txt','r') as file:
    text=file.read().lower()

# tokenize the text- creating indexes for words
tokenizer=Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index)+1
print(total_words)

4818


In [16]:
# create input sequences
inputsequences=[]
for line in text.split('\n'):
    token_list=tokenizer.texts_to_sequences([line])[0]
    for i in range(1,len(token_list)):
        n_gram_sequence=token_list[:i+1]
        inputsequences.append(n_gram_sequence)

In [18]:
## pad sequences
max_sequence_len=max([len(x) for x in inputsequences])

inputsequences=np.array(pad_sequences(inputsequences,maxlen=max_sequence_len,padding='pre'))
inputsequences

array([[   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       [   0,    0,    0, ...,  687,    4,   45],
       ...,
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4],
       [   0,    0,    0, ..., 1047,    4,  193]],
      shape=(25732, 14), dtype=int32)

In [27]:
## create predictors and label
import tensorflow as tf
x,y = inputsequences[:,:-1],inputsequences[:,-1]


In [28]:
x

array([[   0,    0,    0, ...,    0,    0,    1],
       [   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       ...,
       [   0,    0,    0, ...,  687,    4,   45],
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4]],
      shape=(25732, 13), dtype=int32)

In [29]:
y

array([ 687,    4,   45, ..., 1047,    4,  193],
      shape=(25732,), dtype=int32)

In [30]:
y= tf.keras.utils.to_categorical(y,num_classes=total_words)

In [31]:
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(25732, 4818))

In [32]:
X_train,X_test, y_train, y_test = train_test_split(x,y,  test_size=0.2)

In [35]:
# Train our LSTM rnn

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout

# model definiton
model=Sequential()
model.add(Embedding(total_words,100,input_length=max_sequence_len))
model.add(LSTM(150,return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words,activation="softmax"))

#compile the model
model.compile(loss="categorical_crossentropy",optimizer='adam',metrics=['accuracy'])
model.summary()

/home/s4ic0/Programming/GenAI/Projects/.venv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [36]:
# trainn the model
history=model.fit(X_train,y_train,epochs=50,validation_data=(X_test,y_test),verbose=1)

Epoch 1/50


W0000 00:00:1775218711.050862  158057 cpu_allocator_impl.cc:82] Allocation of 396714120 exceeds 10% of free system memory.


643/644 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.0315 - loss: 7.1517

W0000 00:00:1775218745.451318  158057 cpu_allocator_impl.cc:82] Allocation of 99192984 exceeds 10% of free system memory.


644/644 ━━━━━━━━━━━━━━━━━━━━ 37s 52ms/step - accuracy: 0.0329 - loss: 6.9186 - val_accuracy: 0.0379 - val_loss: 6.7411
Epoch 2/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 32s 50ms/step - accuracy: 0.0379 - loss: 6.4734 - val_accuracy: 0.0462 - val_loss: 6.8031
Epoch 3/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 33s 51ms/step - accuracy: 0.0454 - loss: 6.3314 - val_accuracy: 0.0552 - val_loss: 6.8504
Epoch 4/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 31s 48ms/step - accuracy: 0.0499 - loss: 6.1899 - val_accuracy: 0.0515 - val_loss: 6.8590
Epoch 5/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 33s 51ms/step - accuracy: 0.0540 - loss: 6.0538 - val_accuracy: 0.0567 - val_loss: 6.8886
Epoch 6/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 26s 40ms/step - accuracy: 0.0622 - loss: 5.9068 - val_accuracy: 0.0676 - val_loss: 6.9246
Epoch 7/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 25s 38ms/step - accuracy: 0.0706 - loss: 5.7542 - val_accuracy: 0.0666 - val_loss: 7.0016
Epoch 8/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 24s 37ms/step - accuracy: 0.0780 - loss: 5.6103 - val_accurac

In [37]:
# function to predict next word

def predict_next_word(model,tokenizer,text,max_sequence_len):
    token_list = tokenizer.texts_to_sequences([text])[0]
    if len(token_list)>=max_sequence_len:
        token_list = token_list[-(max_sequence_len):]
    token_list= pad_sequences([token_list],maxlen=max_sequence_len,padding='pre')
    predicted = model.predict(token_list,verbose=0)
    predicted_word_index=np.argmax(predicted,axis=1)
    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word
    return None

In [38]:
input_text = "To be or not to be"
print(f"input text : {input_text}")
max_sequence_len = model.input_shape[1]
next_word = predict_next_word(model,tokenizer,input_text,max_sequence_len)
print(next_word)

input text : To be or not to be
free


In [40]:
# save the model
model.save('next_word_lstm.keras')

import pickle
with open('tokenizer.pickle','wb') as handle:
    pickle.dump(tokenizer,handle,protocol=pickle.HIGHEST_PROTOCOL)